In [1]:
import polars as pl
from csrio_image2biomass.configs.settings import RAW_DATA_DIR, AUGUMENTED_DATA_DIR

train = pl.read_csv(RAW_DATA_DIR / "train.csv")
test = pl.read_csv(RAW_DATA_DIR / "test.csv")
train

sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
str,str,str,str,str,f64,f64,str,f64
"""ID1011485656__Dry_Clover_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Clover_g""",0.0
"""ID1011485656__Dry_Dead_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Dead_g""",31.9984
"""ID1011485656__Dry_Green_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Green_g""",16.2751
"""ID1011485656__Dry_Total_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Total_g""",48.2735
"""ID1011485656__GDM_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""GDM_g""",16.275
…,…,…,…,…,…,…,…,…
"""ID983582017__Dry_Clover_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Clover_g""",0.0
"""ID983582017__Dry_Dead_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Dead_g""",0.0
"""ID983582017__Dry_Green_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Green_g""",40.94


In [2]:
train = train.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target").join(train.select("image_path", "State", "Species").unique(), on="image_path")
train

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,State,Species
str,f64,f64,f64,f64,f64,str,str
"""train/ID1510574031.jpg""",7.2336,14.4672,19.4992,41.2,26.7328,"""Vic""","""Phalaris_Clover_Ryegrass_Barle…"
"""train/ID761508093.jpg""",0.0,30.8,20.5333,51.3333,20.5333,"""Vic""","""Mixed"""
"""train/ID1623964968.jpg""",10.0977,21.4983,19.5439,51.1398,29.6415,"""Tas""","""Ryegrass_Clover"""
"""train/ID1036339023.jpg""",23.0755,2.6135,32.191,57.88,55.2665,"""Vic""","""Phalaris_Clover"""
"""train/ID1343327476.jpg""",3.1429,3.1429,37.7143,44.0,40.8571,"""Vic""","""Phalaris_Clover"""
…,…,…,…,…,…,…,…
"""train/ID1244346858.jpg""",0.0,30.9343,49.2657,80.2,49.2657,"""NSW""","""Fescue"""
"""train/ID1574125908.jpg""",0.0,3.5379,43.1621,46.7,43.1621,"""NSW""","""Ryegrass"""
"""train/ID1545077474.jpg""",0.0,19.2,8.8,28.0,8.8,"""Tas""","""Ryegrass_Clover"""


In [3]:
from sklearn.model_selection import train_test_split
train, val = train_test_split(
    train, test_size=0.1, random_state=42, shuffle=True, stratify=train[['State', 'Species']]
)

In [4]:
train_cleaned = train.select("image_path", "Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g") 
val_cleaned = val.select("image_path", "Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g") 
test_df = test.with_columns(pl.lit(0).alias("target"))
test_cleaned = test_df.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target")
train_cleaned.head(10)

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID875119737.jpg""",5.8,5.8,23.2,34.8,29.0
"""train/ID249042826.jpg""",10.6616,3.8769,1.8,16.3385,12.4615
"""train/ID1831254380.jpg""",71.7865,15.758,0.8754,88.42,72.662
"""train/ID135365668.jpg""",2.4,1.8667,17.3333,21.6,19.7333
"""train/ID956512130.jpg""",0.0,1.9,2.4,4.3,2.4
"""train/ID668475812.jpg""",0.0,52.9301,34.0699,87.0,34.0699
"""train/ID980538882.jpg""",0.0,1.1457,91.6543,92.8,91.6543
"""train/ID1035947949.jpg""",0.4343,23.2239,10.5261,34.1844,10.9605
"""train/ID1509266870.jpg""",2.775,2.775,27.75,33.3,30.525


In [5]:
test_cleaned

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i32,i32,i32,i32,i32
"""test/ID1001187975.jpg""",0,0,0,0,0


In [6]:
import cv2
import albumentations as A
from tqdm.auto import tqdm

def augment_data(df: pl.DataFrame):
    augmented_images_dir = AUGUMENTED_DATA_DIR / 'train'
    augmented_images_dir.mkdir(exist_ok=True)

    # Define transformations
    no_transform = A.Compose([])
    h_flip = A.Compose([A.HorizontalFlip(p=1.0)])
    v_flip = A.Compose([A.VerticalFlip(p=1.0)])
    hv_flip = A.Compose([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)])

    transforms = [
        (no_transform, ""),
        (h_flip, "_hflip"),
        (v_flip, "_vflip"),
        (hv_flip, "_hvflip")
    ]

    # Create list to store augmented data
    augmented_data = []

    # Process each image in df
    for row in tqdm(df.iter_rows(named=True), total=len(df)):
        img_path = RAW_DATA_DIR / row['image_path']
        image = cv2.imread(str(img_path))
        
        if image is None:
            continue
        
        # Get the base filename without extension
        base_name = row['image_path'].replace('train/', '').replace('.jpg', '')
        
        for transform, suffix in transforms:
            # Apply transformation
            augmented = transform(image=image)['image']
            
            # Save augmented image
            aug_filename = f"{base_name}{suffix}.jpg"
            aug_path = augmented_images_dir / aug_filename
            cv2.imwrite(str(aug_path), augmented)
            
            if transform == no_transform:
                continue
            
            # Create new row with augmented image path
            new_row = {
                'image_path': f"train/{aug_filename}",
                'Dry_Clover_g': row['Dry_Clover_g'],
                'Dry_Dead_g': row['Dry_Dead_g'],
                'Dry_Green_g': row['Dry_Green_g'],
                'Dry_Total_g': row['Dry_Total_g'],
                'GDM_g': row['GDM_g']
            }
            augmented_data.append(new_row)

    # Create new dataframe with augmented data
    train_augmented = pl.DataFrame(augmented_data)

    # Combine original and augmented data
    df_expanded = pl.concat([df, train_augmented])

    print(f"Original dataset size: {len(df)}")
    print(f"Expanded dataset size: {len(df_expanded)}")

    return df_expanded


train_expanded = augment_data(train_cleaned)
val_expanded = augment_data(val_cleaned)

  0%|          | 0/321 [00:00<?, ?it/s]

Original dataset size: 321
Expanded dataset size: 1284


  0%|          | 0/36 [00:00<?, ?it/s]

Original dataset size: 36
Expanded dataset size: 144


In [7]:
train_expanded.write_csv(AUGUMENTED_DATA_DIR / "train.csv")
val_expanded.write_csv(AUGUMENTED_DATA_DIR / "val.csv")
test_cleaned.write_csv(AUGUMENTED_DATA_DIR / "test.csv")